# Prefix LM Exp 05: Hero Run — 6 Datasets, 17 Conditions, 3 Wrong-Answer Types

## Motivation

Exp 04h proved that new evaluation methodologies reveal signal hidden by mean NLL:
- **Token stratification**: Oracle beats random on hard tokens (d gradient +0.090 Q1->Q4)
- **Contrastive evaluation**: Oracle increases discrimination (d=+0.193, ***),
  concentrated on hard samples (Q4: d=+0.475, ***)
- **Vocabulary bridging**: oracle_plus_vocab (d=+0.311 vs random) and pointer
  instructions (d=+0.250) are the only approaches that beat the structural baseline.

But 04h only tested 5 conditions on MS MARCO. This hero run applies the full evaluation
battery across **17 conditions** (including LLM-generated surrogates, vocabulary bridge
dose-response, and pointer instructions) and **6 datasets** (3000 total samples) with
**3 wrong-answer types** (random, hard, LLM-generated) to produce the definitive
surrogate comparison table.

## Datasets (6 x N=500 = 3000 total)

| Dataset | Source | Doc length | Answer type | Why |
|---------|--------|-----------|-------------|-----|
| MS MARCO v1.1 | `microsoft/ms_marco` val | ~60w | ~20w factoid | Established baseline |
| SQuAD v2 | `rajpurkar/squad_v2` val | ~130w | ~3w extractive | Medium docs, extractive |
| neural-bridge | `neural-bridge/rag-dataset-12000` train | ~600w | ~43w generative | Long docs |
| BoolQ | `google/boolq` val | ~100w | 1 token (yes/no) | Pure discrimination |
| TriviaQA | `trivia_qa` rc subset | ~300-500w | ~2w factoid | Long docs, diverse |
| HotpotQA | `hotpot_qa` distractor | ~500w (10 paragraphs) | ~2w multi-hop | Hardest |

## Conditions (17)

| # | Condition | Category | Description |
|---|-----------|----------|-------------|
| 1 | `no_doc` | control | Single-pass [BOS, query, answer] |
| 2 | `bare` | control | Two-pass, no prefix |
| 3 | `random` | structural | 8 random words from WORD_POOL |
| 4 | `oracle` | semantic | Real query as prefix |
| 5 | `oracle_plus_vocab` | semantic | Query + answer-doc overlap words (up to 10) |
| 6 | `pointer` | semantic | "the answer is about [5 overlap keywords]" |
| 7 | `doc_kw10` | extraction | Top-10 TF keywords from document |
| 8 | `instruct` | extraction | "Identify the key facts:" |
| 9 | `llm_query` | LLM | LLM-generated question about passage |
| 10 | `llm_summary` | LLM | LLM-generated 1-sentence summary |
| 11 | `llm_keywords` | LLM | LLM-generated keyword list |
| 12 | `vocab_bridge_1` | dose-response | 1 answer-doc overlap word |
| 13 | `vocab_bridge_3` | dose-response | 3 overlap words |
| 14 | `vocab_bridge_5` | dose-response | 5 overlap words |
| 15 | `vocab_bridge_10` | dose-response | 10 overlap words |
| 16 | `vocab_bridge_15` | dose-response | 15 overlap words |
| 17 | `vocab_bridge_20` | dose-response | 20 overlap words |

BoolQ: skip conditions 12-17 (degenerate -- "yes"/"no" has no content words). Uses 11 conditions.

## Wrong-Answer Pairing (3 types)

1. **Random negatives**: offset (i+250)%N
2. **Hard negatives**: per-dataset topic-matched (Jaccard, same-passage, yes/no flip)
3. **LLM-generated negatives**: Gemma generates plausible-but-wrong answer per sample

## Evaluation Battery (per dataset)

- A: Standard NLL ranking (d vs bare, win%, p-value, Bonferroni)
- B: Token stratification by difficulty (bare NLL quartiles, d(oracle-random) gradient)
- C: Document dependence (no_doc - bare quartiles)
- D: Contrastive AUC -- random negatives
- E: Contrastive AUC -- hard negatives
- F: Contrastive AUC -- LLM-generated negatives
- G: Targeted NLL (top-quartile hard + doc-dependent tokens only)
- H: Vocabulary bridge dose-response curve
- I: Cross-dataset comparison (Kendall's tau, structural fraction, category means)

In [1]:
# Cell 1: Setup
import os
os.umask(0o000)

import sys, json, time, gc, re, copy
import random as pyrandom
import numpy as np
import torch
import torch.nn.functional as F
from pathlib import Path
from collections import Counter
from scipy import stats
from tqdm.auto import tqdm

sys.path.insert(0, "../../..")
from lib.analysis import cohens_d
from lib.data import count_words

SEED = 42
N_SAMPLES = 500

MODEL_NAME = "google/gemma-3-12b-it"

RESULTS_DIR = Path("../../../results/prefix_lm_exp05")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
torch.manual_seed(SEED)
pyrandom.seed(SEED)

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())
HF_TOKEN = os.environ.get("HF_TOKEN")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -- All 17 conditions --
ALL_CONDITIONS = [
    "no_doc", "bare", "random", "oracle", "oracle_plus_vocab",
    "pointer", "doc_kw10", "instruct",
    "llm_query", "llm_summary", "llm_keywords",
    "vocab_bridge_1", "vocab_bridge_3", "vocab_bridge_5",
    "vocab_bridge_10", "vocab_bridge_15", "vocab_bridge_20",
]

# BoolQ uses only 11 conditions (skip dose-response -- "yes"/"no" has no content words)
VOCAB_BRIDGE_CONDITIONS = [
    "vocab_bridge_1", "vocab_bridge_3", "vocab_bridge_5",
    "vocab_bridge_10", "vocab_bridge_15", "vocab_bridge_20",
]
BOOLQ_CONDITIONS = [c for c in ALL_CONDITIONS if c not in VOCAB_BRIDGE_CONDITIONS]

TWO_PASS_CONDITIONS = [c for c in ALL_CONDITIONS if c != "no_doc"]

CONDITION_CATEGORIES = {
    "no_doc": "control", "bare": "control",
    "random": "structural", "oracle": "semantic",
    "oracle_plus_vocab": "semantic", "pointer": "semantic",
    "doc_kw10": "extraction", "instruct": "extraction",
    "llm_query": "LLM", "llm_summary": "LLM", "llm_keywords": "LLM",
    "vocab_bridge_1": "dose-response", "vocab_bridge_3": "dose-response",
    "vocab_bridge_5": "dose-response", "vocab_bridge_10": "dose-response",
    "vocab_bridge_15": "dose-response", "vocab_bridge_20": "dose-response",
}

DATASET_NAMES = ["msmarco", "squad", "neuralbridge", "boolq", "triviaqa", "hotpotqa"]

STOP_WORDS = {
    'a', 'an', 'the', 'is', 'are', 'was', 'were', 'be', 'been', 'being',
    'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'could',
    'should', 'may', 'might', 'can', 'shall', 'to', 'of', 'in', 'for',
    'on', 'with', 'at', 'by', 'from', 'as', 'into', 'through', 'during',
    'before', 'after', 'above', 'below', 'between', 'and', 'but', 'or',
    'not', 'no', 'if', 'then', 'than', 'so', 'up', 'out', 'about',
    'what', 'which', 'who', 'whom', 'this', 'that', 'these', 'those',
    'it', 'its', 'i', 'me', 'my', 'we', 'our', 'you', 'your', 'he',
    'him', 'his', 'she', 'her', 'they', 'them', 'their', 'how', 'when',
    'where', 'why', 'much', 'many', 'some', 'any', 'all', 'each',
    'does', 'also', 'just', 'more', 'most', 'very', 'too', 'only',
}

WORD_POOL = [
    "computer", "mountain", "hospital", "children", "building", "national",
    "business", "research", "students", "american", "possible", "economic",
    "personal", "together", "products", "services", "actually", "remember",
    "practice", "training", "industry", "complete", "critical", "function",
    "language", "standard", "material", "original", "physical", "security",
    "interest", "problems", "consider", "response", "pressure", "politics",
    "movement", "evidence", "southern", "northern", "exchange", "decision",
    "position", "increase", "describe", "military", "required", "approach",
    "strategy", "customer", "resource", "employee", "audience", "location",
    "property", "cultural", "activity", "strength", "analysis", "powerful",
    "election", "argument", "campaign", "maintain", "question", "behavior",
    "majority", "solution", "software", "consumer", "creative", "reaction",
    "european", "delivery", "organize", "involved", "relative", "learning",
    "positive", "numerous", "familiar", "engineer", "platform", "indicate",
    "previous", "pleasure", "opposite", "magazine", "document", "religion",
    "scenario", "workshop", "minority", "guidance", "estimate", "recently",
    "surprise", "champion", "pleasant", "grateful", "moderate", "boundary",
]

INSTRUCT_PREFIX = "Identify the key facts:"


def content_words(text):
    words = re.sub(r'[^\w\s]', '', text.lower()).split()
    return [w for w in words if w not in STOP_WORDS and len(w) > 2]


def jaccard(set_a, set_b):
    union = set_a | set_b
    return len(set_a & set_b) / len(union) if union else 0.0


def extract_keywords(text, n=10):
    words = re.sub(r'[^\w\s]', '', text.lower()).split()
    filtered = [w for w in words if w not in STOP_WORDS and len(w) > 2]
    counts = Counter(filtered)
    return " ".join(w for w, _ in counts.most_common(n))


def get_conditions_for_dataset(ds_name):
    if ds_name == "boolq":
        return BOOLQ_CONDITIONS
    return ALL_CONDITIONS


ANSWER_TYPES = ['correct', 'wrong_random', 'wrong_hard', 'wrong_llm']

def get_answer_types_for_dataset(ds_name):
    # BoolQ: skip wrong_llm (natural hard neg is sufficient)
    if ds_name == "boolq":
        return ['correct', 'wrong_random', 'wrong_hard']
    return ANSWER_TYPES


print(f"Prefix LM Exp 05: Hero Run -- 6 Datasets, 17 Conditions, 3 Wrong-Answer Types")
print(f"N: {N_SAMPLES} per dataset, Total: {N_SAMPLES * len(DATASET_NAMES)}")
print(f"Datasets: {DATASET_NAMES}")
print(f"DEVICE: {DEVICE}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"\nAll conditions ({len(ALL_CONDITIONS)}): {ALL_CONDITIONS}")
print(f"BoolQ conditions ({len(BOOLQ_CONDITIONS)}): {BOOLQ_CONDITIONS}")
print(f"Answer types: {ANSWER_TYPES}")


Prefix LM Exp 05: Hero Run -- 6 Datasets, 17 Conditions, 3 Wrong-Answer Types
N: 500 per dataset, Total: 3000
Datasets: ['msmarco', 'squad', 'neuralbridge', 'boolq', 'triviaqa', 'hotpotqa']
DEVICE: cuda
GPU: NVIDIA A100-SXM4-40GB
GPU memory: 42.3 GB

All conditions (17): ['no_doc', 'bare', 'random', 'oracle', 'oracle_plus_vocab', 'pointer', 'doc_kw10', 'instruct', 'llm_query', 'llm_summary', 'llm_keywords', 'vocab_bridge_1', 'vocab_bridge_3', 'vocab_bridge_5', 'vocab_bridge_10', 'vocab_bridge_15', 'vocab_bridge_20']
BoolQ conditions (11): ['no_doc', 'bare', 'random', 'oracle', 'oracle_plus_vocab', 'pointer', 'doc_kw10', 'instruct', 'llm_query', 'llm_summary', 'llm_keywords']
Answer types: ['correct', 'wrong_random', 'wrong_hard', 'wrong_llm']


In [2]:
# Load all 6 dataset checkpoints
print("Loading all checkpoints...")

all_datasets = {}
for ds_name in DATASET_NAMES:
    cp_path = RESULTS_DIR / ds_name / "checkpoint.json"
    if cp_path.exists():
        ckpt = json.loads(cp_path.read_text())
        results = ckpt["results"]
        globals()[f"{ds_name}_results"] = results
        all_datasets[ds_name] = results
        print(f"  {ds_name}: loaded {len(results)} results from checkpoint")
    else:
        print(f"  WARNING: {ds_name} checkpoint not found at {cp_path}")

# Populate sample list placeholders (None — sample objects not needed for NLL analysis)
for ds_name in DATASET_NAMES:
    globals()[f"{ds_name}_samples"] = None

print(f"\nLoaded {len(all_datasets)} datasets, {sum(len(v) for v in all_datasets.values())} total samples.")


Loading all checkpoints...


  msmarco: loaded 500 results from checkpoint
  squad: loaded 500 results from checkpoint


  neuralbridge: loaded 500 results from checkpoint
  boolq: loaded 500 results from checkpoint


  triviaqa: loaded 500 results from checkpoint
  hotpotqa: loaded 500 results from checkpoint

Loaded 6 datasets, 3000 total samples.


In [3]:
# Cell 13: Analysis A -- Standard NLL ranking per dataset
print("=" * 70)
print("ANALYSIS A: STANDARD NLL RANKING")
print("=" * 70)

all_datasets = {
    'msmarco': msmarco_results,
    'squad': squad_results,
    'neuralbridge': neuralbridge_results,
    'boolq': boolq_results,
    'triviaqa': triviaqa_results,
    'hotpotqa': hotpotqa_results,
}

# Storage for cross-dataset comparison
dataset_summaries = {}

for ds_name, results in all_datasets.items():
    N = len(results)
    conditions = get_conditions_for_dataset(ds_name)
    print(f"\n{'='*50}")
    print(f"  Dataset: {ds_name} (N={N}, {len(conditions)} conditions)")
    print(f"{'='*50}")

    # Gather per-sample mean NLLs (correct answer)
    nll_correct = {}
    for cn in conditions:
        nll_correct[cn] = np.array([r[f'nll_{cn}_correct'] for r in results])

    # d vs bare, win%, p-value, Bonferroni
    n_tests = len(conditions) - 1  # exclude bare
    bonferroni = 0.05 / n_tests

    print(f"\n  {'Condition':<25} {'Category':<14} {'NLL':>8} {'d_vs_bare':>10}"
          f" {'win%':>6} {'p':>12} {'sig':>5} {'Bonf':>5}")
    print(f"  {'-'*90}")

    cond_stats = {}
    for cn in conditions:
        cat = CONDITION_CATEGORIES[cn]
        nll_m = nll_correct[cn].mean()
        if cn == 'bare':
            d_vs_bare = 0.0
            win_pct = 0.5
            p_val = 1.0
        else:
            diff = nll_correct['bare'] - nll_correct[cn]
            d_vs_bare = cohens_d(diff)
            win_pct = (diff > 0).mean()
            _, p_val = stats.ttest_1samp(diff, 0)
        sig = ('***' if p_val < 0.001 else '**' if p_val < 0.01
               else '*' if p_val < 0.05 else 'ns')
        bonf_sig = '*' if p_val < bonferroni else ''
        print(f"  {cn:<25} {cat:<14} {nll_m:>8.4f} {d_vs_bare:>+10.3f}"
              f" {win_pct:>6.1%} {p_val:>12.2e} {sig:>5} {bonf_sig:>5}")

        cond_stats[cn] = {
            'nll_correct': float(nll_m),
            'd_correct_vs_bare': float(d_vs_bare),
            'win_pct': float(win_pct),
            'p_val': float(p_val),
        }

    # Structural fraction
    d_oracle = cond_stats['oracle']['d_correct_vs_bare']
    d_random = cond_stats['random']['d_correct_vs_bare']
    sf = d_random / d_oracle if d_oracle != 0 else float('nan')
    print(f"\n  Structural fraction: d_random/d_oracle = {d_random:+.3f}/{d_oracle:+.3f} = {sf:.0%}")

    dataset_summaries[ds_name] = {
        'n_samples': N,
        'conditions': conditions,
        'nll_correct': nll_correct,
        'cond_stats': cond_stats,
        'structural_fraction': float(sf),
    }

print(f"\n\nAnalysis A complete for all {len(all_datasets)} datasets.")


ANALYSIS A: STANDARD NLL RANKING

  Dataset: msmarco (N=500, 17 conditions)

  Condition                 Category            NLL  d_vs_bare   win%            p   sig  Bonf
  ------------------------------------------------------------------------------------------
  no_doc                    control          5.7512     -1.232   0.6%    3.20e-102   ***     *
  bare                      control          2.9572     +0.000  50.0%     1.00e+00    ns      
  random                    structural       2.2979     +0.475  86.0%     7.13e-24   ***     *
  oracle                    semantic         1.9678     +0.452  83.2%     6.03e-22   ***     *
  oracle_plus_vocab         semantic         1.8939     +0.474  88.4%     7.82e-24   ***     *
  pointer                   semantic         2.0894     +0.521  94.0%     6.32e-28   ***     *
  doc_kw10                  extraction       2.1218     +0.461  87.0%     9.52e-23   ***     *
  instruct                  extraction       2.4328     +0.471  80.0% 

  oracle                    semantic         3.0939     +0.861  86.4%     3.13e-62   ***     *
  oracle_plus_vocab         semantic         2.8758     +0.883  86.6%     1.35e-64   ***     *
  pointer                   semantic         3.8081     +0.440  69.4%     5.70e-21   ***     *
  doc_kw10                  extraction       3.8281     +0.506  70.8%     1.39e-26   ***     *
  instruct                  extraction       4.4848     +0.033  48.8%     4.66e-01    ns      
  llm_query                 LLM              3.4834     +0.668  77.6%     5.99e-42   ***     *
  llm_summary               LLM              4.1116     +0.286  63.8%     3.79e-10   ***     *
  llm_keywords              LLM              3.8299     +0.543  72.0%     6.31e-30   ***     *
  vocab_bridge_1            dose-response    4.5197     +0.010  51.4%     8.18e-01    ns      
  vocab_bridge_3            dose-response    4.3549     +0.138  58.4%     2.13e-03    **     *
  vocab_bridge_5            dose-response    4.355

In [4]:
# Cell 14: Analysis B -- Token stratification by difficulty quartiles
print("=" * 70)
print("ANALYSIS B: TOKEN STRATIFICATION BY DIFFICULTY")
print("=" * 70)

for ds_name, results in all_datasets.items():
    N = len(results)
    conditions = get_conditions_for_dataset(ds_name)
    print(f"\n{'='*50}")
    print(f"  Dataset: {ds_name} (N={N})")
    print(f"{'='*50}")

    # Gather per-token NLLs for correct answers
    tok_data = {cn: [] for cn in conditions}
    tok_ids_flat = []

    for r in results:
        n_a = len(r['correct_answer_ids'])
        for cn in conditions:
            tok_data[cn].extend(r[f'token_nlls_{cn}_correct'])
        tok_ids_flat.extend(r['correct_answer_ids'])

    for cn in conditions:
        tok_data[cn] = np.array(tok_data[cn])
    tok_ids_flat = np.array(tok_ids_flat)
    n_tokens = len(tok_ids_flat)

    print(f"\n  Total tokens pooled: {n_tokens}")
    print(f"  Mean tokens/sample: {n_tokens / N:.1f}")

    # Difficulty quartiles (bare NLL)
    difficulty = tok_data['bare']
    q25, q50, q75 = np.percentile(difficulty, [25, 50, 75])
    quartile = np.digitize(difficulty, bins=[q25, q50, q75])

    q_labels = ["Q1 (easy)", "Q2", "Q3", "Q4 (hard)"]
    print(f"\n  {'Quartile':<12} {'N':>6} {'bare':>8} {'random':>8}"
          f" {'oracle':>8} {'d(o-r)':>8} {'d(o-b)':>8}")
    print(f"  {'-'*60}")

    d_or_by_q = []
    for q in range(4):
        mask = quartile == q
        n_tok = mask.sum()
        bare_m = tok_data['bare'][mask].mean()
        rand_m = tok_data['random'][mask].mean()
        orac_m = tok_data['oracle'][mask].mean()
        d_or = cohens_d(tok_data['random'][mask] - tok_data['oracle'][mask])
        d_ob = cohens_d(tok_data['bare'][mask] - tok_data['oracle'][mask])
        d_or_by_q.append(d_or)
        print(f"  {q_labels[q]:<12} {n_tok:>6} {bare_m:>8.3f} {rand_m:>8.3f}"
              f" {orac_m:>8.3f} {d_or:>+8.3f} {d_ob:>+8.3f}")

    gradient = d_or_by_q[3] - d_or_by_q[0]
    print(f"\n  d(oracle-random) gradient (Q4-Q1): {gradient:+.3f}")

    dataset_summaries[ds_name]['tok_data'] = tok_data
    dataset_summaries[ds_name]['quartile'] = quartile
    dataset_summaries[ds_name]['d_gradient_q4_q1'] = float(gradient)
    dataset_summaries[ds_name]['n_tokens'] = n_tokens

print(f"\n\nAnalysis B complete.")


ANALYSIS B: TOKEN STRATIFICATION BY DIFFICULTY

  Dataset: msmarco (N=500)

  Total tokens pooled: 9595
  Mean tokens/sample: 19.2

  Quartile          N     bare   random   oracle   d(o-r)   d(o-b)
  ------------------------------------------------------------
  Q1 (easy)      2396    0.000    0.005    0.004   +0.007   -0.040
  Q2             2401    0.002    0.005    0.009   -0.050   -0.071
  Q3             2399    0.042    0.084    0.126   -0.100   -0.203
  Q4 (hard)      2399    4.154    3.473    3.288   +0.097   +0.294

  d(oracle-random) gradient (Q4-Q1): +0.090

  Dataset: squad (N=500)

  Total tokens pooled: 2727
  Mean tokens/sample: 5.5

  Quartile          N     bare   random   oracle   d(o-r)   d(o-b)
  ------------------------------------------------------------
  Q1 (easy)       680    0.001    0.001    0.002   -0.035   -0.062
  Q2              682    0.008    0.017    0.023   -0.049   -0.137
  Q3              682    0.185    0.239    0.291   -0.076   -0.178
  Q4 (hard) 

In [5]:
# Cell 15: Analysis C -- Document dependence quartiles
print("=" * 70)
print("ANALYSIS C: DOCUMENT DEPENDENCE")
print("=" * 70)

for ds_name, results in all_datasets.items():
    conditions = get_conditions_for_dataset(ds_name)
    tok_data = dataset_summaries[ds_name]['tok_data']
    print(f"\n{'='*50}")
    print(f"  Dataset: {ds_name}")
    print(f"{'='*50}")

    doc_dep = tok_data['no_doc'] - tok_data['bare']
    dq25, dq50, dq75 = np.percentile(doc_dep, [25, 50, 75])
    dep_quartile = np.digitize(doc_dep, bins=[dq25, dq50, dq75])

    dep_labels = ["Q1 (doc-indep)", "Q2", "Q3", "Q4 (doc-dep)"]
    print(f"\n  {'Quartile':<16} {'N':>6} {'dep':>8} {'d(o-r)':>8} {'d(o-b)':>8}")
    print(f"  {'-'*50}")

    for q in range(4):
        mask = dep_quartile == q
        dep_m = doc_dep[mask].mean()
        d_or = cohens_d(tok_data['random'][mask] - tok_data['oracle'][mask])
        d_ob = cohens_d(tok_data['bare'][mask] - tok_data['oracle'][mask])
        print(f"  {dep_labels[q]:<16} {mask.sum():>6} {dep_m:>+8.3f}"
              f" {d_or:>+8.3f} {d_ob:>+8.3f}")

    dataset_summaries[ds_name]['dep_quartile'] = dep_quartile

print(f"\n\nAnalysis C complete.")


ANALYSIS C: DOCUMENT DEPENDENCE

  Dataset: msmarco

  Quartile              N      dep   d(o-r)   d(o-b)
  --------------------------------------------------
  Q1 (doc-indep)     2399   -0.167   +0.036   +0.048
  Q2                 2398   +0.207   -0.029   -0.015
  Q3                 2399   +1.798   -0.048   +0.032
  Q4 (doc-dep)       2399   +6.439   +0.109   +0.277

  Dataset: squad

  Quartile              N      dep   d(o-r)   d(o-b)
  --------------------------------------------------
  Q1 (doc-indep)      682   -0.207   +0.192   +0.230
  Q2                  681   +0.708   +0.113   +0.205
  Q3                  682   +3.314   +0.139   +0.343
  Q4 (doc-dep)        682   +9.299   +0.139   +0.487

  Dataset: neuralbridge

  Quartile              N      dep   d(o-r)   d(o-b)
  --------------------------------------------------
  Q1 (doc-indep)     6712   -0.698   -0.210   -0.287
  Q2                 6712   +0.052   -0.191   -0.276
  Q3                 6710   +1.167   -0.249   -0.308
 


  Quartile              N      dep   d(o-r)   d(o-b)
  --------------------------------------------------
  Q1 (doc-indep)      367   -0.660   +0.220   +0.193
  Q2                  366   +0.001   +0.070   -0.003
  Q3                  366   +2.332   +0.560   +0.609
  Q4 (doc-dep)        368   +9.509   +0.921   +0.874

  Dataset: hotpotqa

  Quartile              N      dep   d(o-r)   d(o-b)
  --------------------------------------------------
  Q1 (doc-indep)      484   -0.791   +0.149   +0.179
  Q2                  483   +0.093   +0.021   -0.201
  Q3                  484   +2.791   +0.246   +0.326
  Q4 (doc-dep)        484   +9.503   +0.510   +0.667


Analysis C complete.


In [6]:
# Cell 16: Analysis D -- Contrastive AUC (random negatives)
print("=" * 70)
print("ANALYSIS D: CONTRASTIVE AUC -- RANDOM NEGATIVES")
print("=" * 70)

for ds_name, results in all_datasets.items():
    N = len(results)
    conditions = get_conditions_for_dataset(ds_name)
    print(f"\n{'='*50}")
    print(f"  Dataset: {ds_name} (N={N})")
    print(f"{'='*50}")

    nll_correct = dataset_summaries[ds_name]['nll_correct']
    nll_wrong_rand = {}
    for cn in conditions:
        nll_wrong_rand[cn] = np.array([r[f'nll_{cn}_wrong_random'] for r in results])

    gap_rand = {}
    for cn in conditions:
        gap_rand[cn] = nll_wrong_rand[cn] - nll_correct[cn]

    print(f"\n  {'Condition':<25} {'mean_gap':>10} {'AUC':>8} {'d(gap)':>8}"
          f" {'p':>12} {'sig':>5}")
    print(f"  {'-'*72}")

    for cn in conditions:
        g = gap_rand[cn]
        auc = (g > 0).mean()
        d_gap = cohens_d(g)
        _, p_gap = stats.ttest_1samp(g, 0)
        sig = ('***' if p_gap < 0.001 else '**' if p_gap < 0.01
               else '*' if p_gap < 0.05 else 'ns')
        print(f"  {cn:<25} {g.mean():>+10.3f} {auc:>8.1%} {d_gap:>+8.3f}"
              f" {p_gap:>12.2e} {sig:>5}")

    # Oracle vs random discrimination
    diff_d = gap_rand['oracle'] - gap_rand['random']
    d_d = cohens_d(diff_d)
    _, p_d = stats.ttest_1samp(diff_d, 0)
    sig_d = ('***' if p_d < 0.001 else '**' if p_d < 0.01
             else '*' if p_d < 0.05 else 'ns')
    print(f"\n  Oracle vs random discrimination: d={d_d:+.3f}, p={p_d:.2e} {sig_d}")

    dataset_summaries[ds_name]['gap_rand'] = gap_rand
    dataset_summaries[ds_name]['d_discrim_rand'] = float(d_d)
    dataset_summaries[ds_name]['nll_wrong_rand'] = nll_wrong_rand

    # Per-condition contrastive stats
    for cn in conditions:
        g = gap_rand[cn]
        dataset_summaries[ds_name]['cond_stats'][cn]['auc_rand'] = float((g > 0).mean())
        dataset_summaries[ds_name]['cond_stats'][cn]['d_gap_rand'] = float(cohens_d(g))

print(f"\n\nAnalysis D complete.")


ANALYSIS D: CONTRASTIVE AUC -- RANDOM NEGATIVES

  Dataset: msmarco (N=500)

  Condition                   mean_gap      AUC   d(gap)            p   sig
  ------------------------------------------------------------------------
  no_doc                        +1.533    62.2%   +0.185     4.23e-05   ***
  bare                          +3.205    83.0%   +0.526     2.18e-28   ***
  random                        +3.205    85.8%   +0.664     1.33e-41   ***
  oracle                        +3.511    87.6%   +0.815     2.82e-57   ***
  oracle_plus_vocab             +3.653    88.0%   +0.833     3.31e-59   ***
  pointer                       +3.584    87.6%   +0.771     1.52e-52   ***
  doc_kw10                      +3.499    87.2%   +0.759     2.52e-51   ***
  instruct                      +3.583    84.2%   +0.663     2.00e-41   ***
  llm_query                     +3.455    85.8%   +0.685     1.23e-43   ***
  llm_summary                   +3.488    84.8%   +0.630     3.27e-38   ***
  llm_keywor

  bare                          +2.943    99.8%   +2.696    3.17e-231   ***
  random                        +2.837    99.8%   +2.409    4.41e-210   ***
  oracle                        +2.886    99.4%   +2.362    2.03e-206   ***
  oracle_plus_vocab             +2.812    98.8%   +2.156    6.93e-190   ***
  pointer                       +2.938    99.6%   +2.458    8.52e-214   ***
  doc_kw10                      +2.906    99.6%   +2.435    4.53e-212   ***
  instruct                      +2.944    99.8%   +2.439    2.20e-212   ***
  llm_query                     +2.879    99.2%   +2.388    1.69e-208   ***
  llm_summary                   +2.761    98.2%   +2.085    5.46e-184   ***
  llm_keywords                  +2.882    99.8%   +2.395    5.50e-209   ***
  vocab_bridge_1                +2.971    99.8%   +2.597    3.72e-224   ***
  vocab_bridge_3                +2.885    99.8%   +2.453    1.82e-213   ***
  vocab_bridge_5                +2.873    99.8%   +2.388    1.84e-208   ***
  vocab_brid

  no_doc                        +4.826    71.2%   +0.567     3.95e-32   ***
  bare                          +4.155    79.0%   +0.692     2.20e-44   ***
  random                        +3.588    78.0%   +0.639     4.72e-39   ***
  oracle                        +3.741    80.0%   +0.713     1.44e-46   ***
  oracle_plus_vocab             +3.875    81.8%   +0.752     1.21e-50   ***
  pointer                       +4.249    82.2%   +0.786     3.36e-54   ***
  doc_kw10                      +3.606    78.2%   +0.655     1.05e-40   ***
  instruct                      +4.061    78.8%   +0.685     1.15e-43   ***
  llm_query                     +3.253    78.6%   +0.620     3.19e-37   ***
  llm_summary                   +3.519    76.8%   +0.613     1.53e-36   ***
  llm_keywords                  +3.610    78.0%   +0.658     6.34e-41   ***
  vocab_bridge_1                +4.093    81.0%   +0.698     5.43e-45   ***
  vocab_bridge_3                +4.020    80.4%   +0.701     2.60e-45   ***
  vocab_brid

In [7]:
# Cell 17: Analysis E -- Contrastive AUC (hard negatives)
print("=" * 70)
print("ANALYSIS E: CONTRASTIVE AUC -- HARD NEGATIVES")
print("=" * 70)

for ds_name, results in all_datasets.items():
    N = len(results)
    conditions = get_conditions_for_dataset(ds_name)
    print(f"\n{'='*50}")
    print(f"  Dataset: {ds_name} (N={N})")
    print(f"{'='*50}")

    nll_correct = dataset_summaries[ds_name]['nll_correct']
    nll_wrong_hard = {}
    for cn in conditions:
        nll_wrong_hard[cn] = np.array([r[f'nll_{cn}_wrong_hard'] for r in results])

    gap_hard = {}
    for cn in conditions:
        gap_hard[cn] = nll_wrong_hard[cn] - nll_correct[cn]

    print(f"\n  {'Condition':<25} {'mean_gap':>10} {'AUC':>8} {'d(gap)':>8}"
          f" {'p':>12} {'sig':>5}")
    print(f"  {'-'*72}")

    for cn in conditions:
        g = gap_hard[cn]
        auc = (g > 0).mean()
        d_gap = cohens_d(g)
        _, p_gap = stats.ttest_1samp(g, 0)
        sig = ('***' if p_gap < 0.001 else '**' if p_gap < 0.01
               else '*' if p_gap < 0.05 else 'ns')
        print(f"  {cn:<25} {g.mean():>+10.3f} {auc:>8.1%} {d_gap:>+8.3f}"
              f" {p_gap:>12.2e} {sig:>5}")

    # Oracle vs random
    diff_d = gap_hard['oracle'] - gap_hard['random']
    d_d = cohens_d(diff_d)
    _, p_d = stats.ttest_1samp(diff_d, 0)
    sig_d = ('***' if p_d < 0.001 else '**' if p_d < 0.01
             else '*' if p_d < 0.05 else 'ns')
    print(f"\n  Oracle vs random discrimination (hard neg): d={d_d:+.3f}, p={p_d:.2e} {sig_d}")

    # Compare hard vs random negatives
    gap_rand = dataset_summaries[ds_name]['gap_rand']
    auc_rand_bare = (gap_rand['bare'] > 0).mean()
    auc_hard_bare = (gap_hard['bare'] > 0).mean()
    print(f"  AUC(bare): random_neg={auc_rand_bare:.1%} vs hard_neg={auc_hard_bare:.1%}")

    dataset_summaries[ds_name]['gap_hard'] = gap_hard
    dataset_summaries[ds_name]['d_discrim_hard'] = float(d_d)

    for cn in conditions:
        g = gap_hard[cn]
        dataset_summaries[ds_name]['cond_stats'][cn]['auc_hard'] = float((g > 0).mean())
        dataset_summaries[ds_name]['cond_stats'][cn]['d_gap_hard'] = float(cohens_d(g))

print(f"\n\nAnalysis E complete.")


ANALYSIS E: CONTRASTIVE AUC -- HARD NEGATIVES

  Dataset: msmarco (N=500)

  Condition                   mean_gap      AUC   d(gap)            p   sig
  ------------------------------------------------------------------------
  no_doc                        +5.436    67.4%   +0.481     2.26e-24   ***
  bare                          +6.234    83.6%   +0.710     2.98e-46   ***
  random                        +5.968    86.8%   +0.789     1.91e-54   ***
  oracle                        +5.985    87.6%   +0.850     5.02e-61   ***
  oracle_plus_vocab             +6.180    89.0%   +0.861     3.93e-62   ***
  pointer                       +6.222    88.0%   +0.837     1.35e-59   ***
  doc_kw10                      +6.074    87.8%   +0.831     5.51e-59   ***
  instruct                      +6.338    87.0%   +0.792     7.73e-55   ***
  llm_query                     +6.072    86.0%   +0.794     5.62e-55   ***
  llm_summary                   +6.301    86.0%   +0.769     1.98e-52   ***
  llm_keywords

  vocab_bridge_1                +2.755    99.8%   +2.747    8.68e-235   ***
  vocab_bridge_3                +2.697    99.8%   +2.691    7.19e-231   ***
  vocab_bridge_5                +2.688    99.8%   +2.675    8.95e-230   ***
  vocab_bridge_10               +2.717    99.8%   +2.697    2.71e-231   ***
  vocab_bridge_15               +2.697    99.4%   +2.633    9.15e-227   ***
  vocab_bridge_20               +2.672    99.4%   +2.598    3.39e-224   ***

  Oracle vs random discrimination (hard neg): d=+0.100, p=2.65e-02 *
  AUC(bare): random_neg=99.8% vs hard_neg=100.0%

  Dataset: boolq (N=500)

  Condition                   mean_gap      AUC   d(gap)            p   sig
  ------------------------------------------------------------------------
  no_doc                        +0.188    52.4%   +0.072     1.09e-01    ns


  bare                          +0.774    60.2%   +0.309     1.45e-11   ***
  random                        +1.041    69.2%   +0.434     1.61e-20   ***
  oracle                        +2.429    77.6%   +0.749     2.65e-50   ***
  oracle_plus_vocab             +2.287    76.6%   +0.723     1.56e-47   ***
  pointer                       +1.897    75.0%   +0.692     2.19e-44   ***
  doc_kw10                      +1.646    74.2%   +0.588     4.39e-34   ***
  instruct                      +1.003    66.2%   +0.418     2.92e-19   ***
  llm_query                     +1.192    67.8%   +0.468     2.81e-23   ***
  llm_summary                   +1.067    66.0%   +0.417     3.68e-19   ***
  llm_keywords                  +1.389    70.4%   +0.494     1.46e-25   ***

  Oracle vs random discrimination (hard neg): d=+0.552, p=9.59e-31 ***
  AUC(bare): random_neg=25.4% vs hard_neg=60.2%

  Dataset: triviaqa (N=500)

  Condition                   mean_gap      AUC   d(gap)            p   sig
  ------------

  vocab_bridge_1                +3.679    80.6%   +0.805     3.24e-56   ***
  vocab_bridge_3                +3.575    80.8%   +0.811     8.74e-57   ***
  vocab_bridge_5                +3.573    80.8%   +0.810     9.62e-57   ***
  vocab_bridge_10               +3.573    80.8%   +0.810     9.62e-57   ***
  vocab_bridge_15               +3.573    80.8%   +0.810     9.62e-57   ***
  vocab_bridge_20               +3.573    80.8%   +0.810     9.62e-57   ***

  Oracle vs random discrimination (hard neg): d=+0.272, p=2.25e-09 ***
  AUC(bare): random_neg=84.0% vs hard_neg=81.6%

  Dataset: hotpotqa (N=500)

  Condition                   mean_gap      AUC   d(gap)            p   sig
  ------------------------------------------------------------------------
  no_doc                        +4.258    76.4%   +0.574     9.29e-33   ***
  bare                          +3.598    82.2%   +0.746     5.77e-50   ***
  random                        +3.242    78.4%   +0.695     1.00e-44   ***
  oracle       

In [8]:
# Cell 18: Analysis F -- Contrastive AUC (LLM-generated negatives)
print("=" * 70)
print("ANALYSIS F: CONTRASTIVE AUC -- LLM-GENERATED NEGATIVES")
print("=" * 70)

for ds_name, results in all_datasets.items():
    N = len(results)
    conditions = get_conditions_for_dataset(ds_name)
    answer_types = get_answer_types_for_dataset(ds_name)

    if 'wrong_llm' not in answer_types:
        print(f"\n  {ds_name}: skipping (no LLM negatives)")
        dataset_summaries[ds_name]['gap_llm'] = None
        continue

    print(f"\n{'='*50}")
    print(f"  Dataset: {ds_name} (N={N})")
    print(f"{'='*50}")

    nll_correct = dataset_summaries[ds_name]['nll_correct']
    nll_wrong_llm = {}
    for cn in conditions:
        nll_wrong_llm[cn] = np.array([r[f'nll_{cn}_wrong_llm'] for r in results])

    gap_llm = {}
    for cn in conditions:
        gap_llm[cn] = nll_wrong_llm[cn] - nll_correct[cn]

    print(f"\n  {'Condition':<25} {'mean_gap':>10} {'AUC':>8} {'d(gap)':>8}"
          f" {'p':>12} {'sig':>5}")
    print(f"  {'-'*72}")

    for cn in conditions:
        g = gap_llm[cn]
        auc = (g > 0).mean()
        d_gap = cohens_d(g)
        _, p_gap = stats.ttest_1samp(g, 0)
        sig = ('***' if p_gap < 0.001 else '**' if p_gap < 0.01
               else '*' if p_gap < 0.05 else 'ns')
        print(f"  {cn:<25} {g.mean():>+10.3f} {auc:>8.1%} {d_gap:>+8.3f}"
              f" {p_gap:>12.2e} {sig:>5}")

    # Oracle vs random
    diff_d = gap_llm['oracle'] - gap_llm['random']
    d_d = cohens_d(diff_d)
    _, p_d = stats.ttest_1samp(diff_d, 0)
    sig_d = ('***' if p_d < 0.001 else '**' if p_d < 0.01
             else '*' if p_d < 0.05 else 'ns')
    print(f"\n  Oracle vs random discrimination (LLM neg): d={d_d:+.3f}, p={p_d:.2e} {sig_d}")

    # Compare all 3 negative types for bare
    gap_rand = dataset_summaries[ds_name]['gap_rand']
    gap_hard = dataset_summaries[ds_name]['gap_hard']
    print(f"\n  AUC(bare) by negative type:")
    print(f"    Random neg: {(gap_rand['bare'] > 0).mean():.1%}")
    print(f"    Hard neg:   {(gap_hard['bare'] > 0).mean():.1%}")
    print(f"    LLM neg:    {(gap_llm['bare'] > 0).mean():.1%}")

    dataset_summaries[ds_name]['gap_llm'] = gap_llm
    dataset_summaries[ds_name]['d_discrim_llm'] = float(d_d)

    for cn in conditions:
        g = gap_llm[cn]
        dataset_summaries[ds_name]['cond_stats'][cn]['auc_llm'] = float((g > 0).mean())
        dataset_summaries[ds_name]['cond_stats'][cn]['d_gap_llm'] = float(cohens_d(g))

print(f"\n\nAnalysis F complete.")


ANALYSIS F: CONTRASTIVE AUC -- LLM-GENERATED NEGATIVES

  Dataset: msmarco (N=500)

  Condition                   mean_gap      AUC   d(gap)            p   sig
  ------------------------------------------------------------------------
  no_doc                        -1.412    58.8%   -0.253     2.70e-08   ***
  bare                          +0.614    80.2%   +0.154     6.26e-04   ***


  random                        +1.106    82.8%   +0.365     2.75e-15   ***
  oracle                        +1.596    84.8%   +0.683     1.60e-43   ***
  oracle_plus_vocab             +1.666    87.0%   +0.686     8.69e-44   ***
  pointer                       +1.314    84.2%   +0.465     4.31e-23   ***
  doc_kw10                      +1.393    83.4%   +0.504     2.16e-26   ***
  instruct                      +0.924    82.4%   +0.272     2.36e-09   ***
  llm_query                     +1.228    83.8%   +0.382     1.74e-16   ***
  llm_summary                   +0.901    81.4%   +0.262     8.65e-09   ***
  llm_keywords                  +1.375    84.4%   +0.486     8.27e-25   ***
  vocab_bridge_1                +1.140    83.4%   +0.351     2.43e-14   ***
  vocab_bridge_3                +1.210    83.6%   +0.374     6.60e-16   ***
  vocab_bridge_5                +1.218    83.6%   +0.376     4.50e-16   ***
  vocab_bridge_10               +1.212    83.6%   +0.374     5.71e-16   ***
  vocab_brid

  instruct                      +1.603    98.4%   +1.640    9.70e-144   ***
  llm_query                     +1.526    98.2%   +1.452    4.10e-125   ***
  llm_summary                   +1.393    93.6%   +1.257    7.32e-105   ***
  llm_keywords                  +1.599    98.6%   +1.546    1.56e-134   ***
  vocab_bridge_1                +1.673    98.8%   +1.645    3.30e-144   ***
  vocab_bridge_3                +1.607    98.8%   +1.597    1.34e-139   ***
  vocab_bridge_5                +1.601    98.4%   +1.563    3.23e-136   ***
  vocab_bridge_10               +1.592    98.4%   +1.547    1.13e-134   ***
  vocab_bridge_15               +1.566    98.2%   +1.514    2.08e-131   ***
  vocab_bridge_20               +1.530    97.4%   +1.464    2.50e-126   ***

  Oracle vs random discrimination (LLM neg): d=-0.070, p=1.16e-01 ns

  AUC(bare) by negative type:
    Random neg: 99.8%
    Hard neg:   100.0%
    LLM neg:    98.8%

  boolq: skipping (no LLM negatives)

  Dataset: triviaqa (N=500)

  Co

  bare                          -2.443    23.6%   -0.712     2.12e-46   ***
  random                        -1.901    31.8%   -0.594     1.16e-34   ***
  oracle                        -1.160    45.0%   -0.390     4.10e-17   ***
  oracle_plus_vocab             -1.075    48.4%   -0.363     3.99e-15   ***
  pointer                       -1.464    40.0%   -0.489     4.53e-25   ***
  doc_kw10                      -1.589    36.6%   -0.510     6.20e-27   ***
  instruct                      -1.990    30.8%   -0.594     1.22e-34   ***
  llm_query                     -1.384    41.4%   -0.436     1.13e-20   ***
  llm_summary                   -1.749    38.4%   -0.506     1.33e-26   ***
  llm_keywords                  -1.612    37.2%   -0.534     4.29e-29   ***
  vocab_bridge_1                -2.186    26.4%   -0.638     5.49e-39   ***
  vocab_bridge_3                -1.927    32.4%   -0.559     2.14e-31   ***
  vocab_bridge_5                -1.921    32.8%   -0.557     3.40e-31   ***
  vocab_brid

In [9]:
# Cell 19: Analysis G -- Targeted NLL (hard + doc-dependent tokens only)
print("=" * 70)
print("ANALYSIS G: TARGETED NLL (HARD + DOC-DEPENDENT TOKENS)")
print("=" * 70)

for ds_name, results in all_datasets.items():
    N = len(results)
    conditions = get_conditions_for_dataset(ds_name)
    tok_data = dataset_summaries[ds_name]['tok_data']
    quartile = dataset_summaries[ds_name]['quartile']
    dep_quartile = dataset_summaries[ds_name]['dep_quartile']

    print(f"\n{'='*50}")
    print(f"  Dataset: {ds_name} (N={N})")
    print(f"{'='*50}")

    # Targeted tokens: Q4 difficulty AND Q4 doc-dependent
    target_mask = (quartile == 3) & (dep_quartile == 3)
    n_target = target_mask.sum()
    n_total = len(quartile)
    print(f"\n  Targeted tokens: {n_target}/{n_total} ({100*n_target/n_total:.1f}%)")

    if n_target < 50:
        print(f"  Too few targeted tokens, skipping.")
        dataset_summaries[ds_name]['targeted_d'] = {}
        continue

    print(f"\n  {'Condition':<25} {'NLL_target':>12} {'d_vs_bare':>10}"
          f" {'d(o-r)':>8}")
    print(f"  {'-'*58}")

    targeted_d = {}
    for cn in conditions:
        nll_t = tok_data[cn][target_mask].mean()
        d_vs_bare = cohens_d(tok_data['bare'][target_mask] - tok_data[cn][target_mask])
        targeted_d[cn] = float(d_vs_bare)
        print(f"  {cn:<25} {nll_t:>12.4f} {d_vs_bare:>+10.3f}", end="")
        if cn == conditions[-1] or cn == 'oracle':
            d_or = cohens_d(tok_data['random'][target_mask] - tok_data['oracle'][target_mask])
            print(f" {d_or:>+8.3f}")
        else:
            print()

    # Compare targeted vs aggregate
    d_orc_all = cohens_d(tok_data['bare'] - tok_data['oracle'])
    d_orc_target = targeted_d.get('oracle', 0.0)
    print(f"\n  d_oracle: aggregate={d_orc_all:+.3f}, targeted={d_orc_target:+.3f}")
    print(f"  Targeted amplification: {d_orc_target / d_orc_all:.1f}x" if d_orc_all != 0 else "  N/A")

    dataset_summaries[ds_name]['targeted_d'] = targeted_d

print(f"\n\nAnalysis G complete.")


ANALYSIS G: TARGETED NLL (HARD + DOC-DEPENDENT TOKENS)

  Dataset: msmarco (N=500)

  Targeted tokens: 918/9595 (9.6%)

  Condition                   NLL_target  d_vs_bare   d(o-r)
  ----------------------------------------------------------
  no_doc                         13.2427     -2.447
  bare                            6.4419     +0.000
  random                          5.0235     +0.604
  oracle                          4.5215     +0.517   +0.203
  oracle_plus_vocab               4.4463     +0.586
  pointer                         4.5548     +0.714
  doc_kw10                        4.6982     +0.590
  instruct                        5.2640     +0.579
  llm_query                       5.1109     +0.537
  llm_summary                     5.5888     +0.487
  llm_keywords                    4.8526     +0.552
  vocab_bridge_1                  5.3374     +0.538
  vocab_bridge_3                  4.8854     +0.552
  vocab_bridge_5                  4.8403     +0.576
  vocab_bridge_10    

In [10]:
# Cell 20: Analysis H -- Vocabulary bridge dose-response curve
print("=" * 70)
print("ANALYSIS H: VOCABULARY BRIDGE DOSE-RESPONSE")
print("=" * 70)

bridge_ns = [1, 3, 5, 10, 15, 20]
bridge_conds = [f"vocab_bridge_{n}" for n in bridge_ns]

all_sample_lists = {}
for _ds in DATASET_NAMES:
    all_sample_lists[_ds] = globals().get(f"{_ds}_samples")

for ds_name, results in all_datasets.items():
    conditions = get_conditions_for_dataset(ds_name)
    if 'vocab_bridge_1' not in conditions:
        print(f"\n  {ds_name}: skipping (no dose-response conditions)")
        dataset_summaries[ds_name]['dose_response'] = None
        continue

    N = len(results)
    nll_correct = dataset_summaries[ds_name]['nll_correct']
    print(f"\n{'='*50}")
    print(f"  Dataset: {ds_name} (N={N})")
    print(f"{'='*50}")

    # Degeneracy check: how many samples have fewer overlap words than requested?
    sample_list = all_sample_lists.get(ds_name)
    if sample_list is not None:
        print(f"\n  Overlap word availability:")
        for n_bridge in bridge_ns:
            key = f'vocab_bridge_{n_bridge}'
            actual_ns = [s[f'{key}_actual_n'] for s in sample_list]
            degen_rate = sum(1 for a in actual_ns if a < n_bridge) / len(actual_ns)
            mean_actual = np.mean(actual_ns)
            print(f"    bridge_{n_bridge}: mean_actual={mean_actual:.1f},"
                  f" degeneracy={degen_rate:.1%}")
    else:
        print(f"\n  Overlap word availability: (skipped — loaded from checkpoint)")

    # Dose-response: d vs bare and d vs random
    print(f"\n  {'Condition':<25} {'d_vs_bare':>10} {'d_vs_random':>12}"
          f" {'NLL':>8}")
    print(f"  {'-'*60}")

    # Include reference conditions
    ref_conds = ['bare', 'random', 'oracle', 'oracle_plus_vocab']
    for cn in ref_conds + bridge_conds:
        if cn not in nll_correct:
            continue
        nll_m = nll_correct[cn].mean()
        d_bare = cohens_d(nll_correct['bare'] - nll_correct[cn])
        d_rand = cohens_d(nll_correct['random'] - nll_correct[cn])
        print(f"  {cn:<25} {d_bare:>+10.3f} {d_rand:>+12.3f} {nll_m:>8.4f}")

    # Also check contrastive AUC dose-response
    gap_rand = dataset_summaries[ds_name]['gap_rand']
    print(f"\n  Contrastive AUC dose-response (random negatives):")
    print(f"  {'Condition':<25} {'AUC':>8} {'d(gap)':>8}")
    print(f"  {'-'*44}")

    for cn in ref_conds + bridge_conds:
        if cn not in gap_rand:
            continue
        g = gap_rand[cn]
        auc = (g > 0).mean()
        d_gap = cohens_d(g)
        print(f"  {cn:<25} {auc:>8.1%} {d_gap:>+8.3f}")

    # Store dose-response data
    dose_data = {}
    for n_bridge in bridge_ns:
        cn = f'vocab_bridge_{n_bridge}'
        if cn in nll_correct:
            dose_data[n_bridge] = {
                'd_vs_bare': float(cohens_d(nll_correct['bare'] - nll_correct[cn])),
                'd_vs_random': float(cohens_d(nll_correct['random'] - nll_correct[cn])),
                'nll': float(nll_correct[cn].mean()),
            }
    dataset_summaries[ds_name]['dose_response'] = dose_data

print(f"\n\nAnalysis H complete.")


ANALYSIS H: VOCABULARY BRIDGE DOSE-RESPONSE

  Dataset: msmarco (N=500)

  Overlap word availability: (skipped — loaded from checkpoint)

  Condition                  d_vs_bare  d_vs_random      NLL
  ------------------------------------------------------------
  bare                          +0.000       -0.475   2.9572
  random                        +0.475       +0.000   2.2979
  oracle                        +0.452       +0.266   1.9678
  oracle_plus_vocab             +0.474       +0.311   1.8939
  vocab_bridge_1                +0.478       -0.155   2.4166
  vocab_bridge_3                +0.553       -0.035   2.3260
  vocab_bridge_5                +0.562       -0.023   2.3160
  vocab_bridge_10               +0.568       -0.016   2.3108
  vocab_bridge_15               +0.566       -0.018   2.3121
  vocab_bridge_20               +0.565       -0.019   2.3127

  Contrastive AUC dose-response (random negatives):
  Condition                      AUC   d(gap)
  ---------------------------

  vocab_bridge_10               +0.138       -0.022   4.3555
  vocab_bridge_15               +0.138       -0.022   4.3555
  vocab_bridge_20               +0.138       -0.022   4.3555

  Contrastive AUC dose-response (random negatives):
  Condition                      AUC   d(gap)
  --------------------------------------------
  bare                         84.0%   +0.868
  random                       80.0%   +0.725
  oracle                       88.2%   +1.012
  oracle_plus_vocab            90.2%   +1.121
  vocab_bridge_1               84.4%   +0.830
  vocab_bridge_3               84.4%   +0.831
  vocab_bridge_5               84.4%   +0.831
  vocab_bridge_10              84.4%   +0.831
  vocab_bridge_15              84.4%   +0.831
  vocab_bridge_20              84.4%   +0.831

  Dataset: hotpotqa (N=500)

  Overlap word availability: (skipped — loaded from checkpoint)

  Condition                  d_vs_bare  d_vs_random      NLL
  -----------------------------------------------------

In [11]:
# Cell 21: Analysis I -- Cross-dataset comparison
print("=" * 70)
print("ANALYSIS I: CROSS-DATASET COMPARISON")
print("=" * 70)

# I1. Full effect size table
print(f"\n--- I1. Effect Size Table (d_correct vs bare) ---\n")
# Use only conditions common to all datasets (exclude dose-response for clean comparison)
common_conds = BOOLQ_CONDITIONS  # 11 conditions available in all datasets

print(f"  {'Condition':<25} {'Category':<14}", end="")
for ds in DATASET_NAMES:
    print(f" {ds:>10}", end="")
print(f" {'Mean':>8}")
print(f"  {'-'*(25+14+10*len(DATASET_NAMES)+8)}")

mean_d_all = {}
for cn in common_conds:
    cat = CONDITION_CATEGORIES[cn]
    print(f"  {cn:<25} {cat:<14}", end="")
    ds_vals = []
    for ds in DATASET_NAMES:
        d = dataset_summaries[ds]['cond_stats'][cn]['d_correct_vs_bare']
        print(f" {d:>+10.3f}", end="")
        ds_vals.append(d)
    mean_v = np.mean(ds_vals)
    mean_d_all[cn] = mean_v
    print(f" {mean_v:>+8.3f}")

# I2. Structural fraction per dataset
print(f"\n--- I2. Structural Fraction ---")
for ds in DATASET_NAMES:
    sf = dataset_summaries[ds]['structural_fraction']
    d_o = dataset_summaries[ds]['cond_stats']['oracle']['d_correct_vs_bare']
    d_r = dataset_summaries[ds]['cond_stats']['random']['d_correct_vs_bare']
    print(f"  {ds:<15} d_oracle={d_o:+.3f}, d_random={d_r:+.3f},"
          f" structural={sf:.0%}")

# I3. Surrogate ranking stability (Kendall's tau)
print(f"\n--- I3. Surrogate Ranking Stability (Kendall's tau) ---")
surr_conds = [cn for cn in common_conds if cn not in ('no_doc', 'bare')]

rankings = {}
for ds in DATASET_NAMES:
    rankings[ds] = [dataset_summaries[ds]['cond_stats'][cn]['d_correct_vs_bare']
                    for cn in surr_conds]

import itertools
for ds_a, ds_b in itertools.combinations(DATASET_NAMES, 2):
    tau, p_tau = stats.kendalltau(rankings[ds_a], rankings[ds_b])
    sig = ('***' if p_tau < 0.001 else '**' if p_tau < 0.01
           else '*' if p_tau < 0.05 else 'ns')
    print(f"  {ds_a} vs {ds_b}: tau={tau:+.3f} (p={p_tau:.3f}) {sig}")

# I4. Category means across datasets
print(f"\n--- I4. Category Means ---")
categories = {
    'control': ['no_doc', 'bare'],
    'structural': ['random'],
    'semantic': ['oracle', 'oracle_plus_vocab', 'pointer'],
    'extraction': ['doc_kw10', 'instruct'],
    'LLM': ['llm_query', 'llm_summary', 'llm_keywords'],
}

print(f"\n  {'Category':<14} {'Conditions':<35}", end="")
for ds in DATASET_NAMES:
    print(f" {ds:>10}", end="")
print(f" {'Mean':>8}")
print(f"  {'-'*(14+35+10*len(DATASET_NAMES)+8)}")

for cat_name, cat_conds in categories.items():
    cond_str = ", ".join(cat_conds)
    print(f"  {cat_name:<14} {cond_str:<35}", end="")
    ds_vals = []
    for ds in DATASET_NAMES:
        mean_d = np.mean([
            dataset_summaries[ds]['cond_stats'][cn]['d_correct_vs_bare']
            for cn in cat_conds if cn in dataset_summaries[ds]['cond_stats']
        ])
        print(f" {mean_d:>+10.3f}", end="")
        ds_vals.append(mean_d)
    print(f" {np.mean(ds_vals):>+8.3f}")

# I5. Difficulty gradient generalization
print(f"\n--- I5. Difficulty Gradient ---")
for ds in DATASET_NAMES:
    g = dataset_summaries[ds]['d_gradient_q4_q1']
    print(f"  {ds:<15} gradient={g:+.3f}")

# I6. Discrimination comparison across datasets and negative types
print(f"\n--- I6. Discrimination (oracle vs random) by Negative Type ---")
print(f"  {'Dataset':<15} {'Random':>10} {'Hard':>10} {'LLM':>10}")
print(f"  {'-'*48}")

for ds in DATASET_NAMES:
    d_rand = dataset_summaries[ds].get('d_discrim_rand', float('nan'))
    d_hard = dataset_summaries[ds].get('d_discrim_hard', float('nan'))
    d_llm = dataset_summaries[ds].get('d_discrim_llm', float('nan'))
    print(f"  {ds:<15} {d_rand:>+10.3f} {d_hard:>+10.3f}", end="")
    if np.isnan(d_llm):
        print(f" {'N/A':>10}")
    else:
        print(f" {d_llm:>+10.3f}")

# I7. Contrastive AUC heatmap (all conditions x all datasets, random negatives)
print(f"\n--- I7. AUC Heatmap (random negatives) ---\n")
print(f"  {'Condition':<25}", end="")
for ds in DATASET_NAMES:
    print(f" {ds:>10}", end="")
print()
print(f"  {'-'*(25+10*len(DATASET_NAMES))}")

for cn in common_conds:
    print(f"  {cn:<25}", end="")
    for ds in DATASET_NAMES:
        auc = dataset_summaries[ds]['cond_stats'][cn].get('auc_rand', float('nan'))
        if np.isnan(auc):
            print(f" {'N/A':>10}", end="")
        else:
            print(f" {auc:>10.1%}", end="")
    print()

print(f"\n\nAnalysis I complete.")


ANALYSIS I: CROSS-DATASET COMPARISON

--- I1. Effect Size Table (d_correct vs bare) ---

  Condition                 Category          msmarco      squad neuralbridge      boolq   triviaqa   hotpotqa     Mean
  -----------------------------------------------------------------------------------------------------------
  no_doc                    control            -1.232     -1.592     -1.743     -1.942     -1.146     -1.235   -1.482
  bare                      control            +0.000     +0.000     +0.000     +0.000     +0.000     +0.000   +0.000
  random                    structural         +0.475     +0.658     -0.480     +1.518     +0.160     +0.292   +0.437
  oracle                    semantic           +0.452     +0.823     -0.514     +1.868     +0.861     +0.598   +0.682
  oracle_plus_vocab         semantic           +0.474     +1.063     -0.590     +1.517     +0.883     +0.566   +0.652
  pointer                   semantic           +0.521     +0.933     -0.451     +1.491     

  msmarco vs squad: tau=+0.333 (p=0.260) ns
  msmarco vs neuralbridge: tau=+0.444 (p=0.119) ns
  msmarco vs boolq: tau=-0.222 (p=0.477) ns
  msmarco vs triviaqa: tau=+0.000 (p=1.000) ns
  msmarco vs hotpotqa: tau=+0.278 (p=0.358) ns
  squad vs neuralbridge: tau=+0.000 (p=1.000) ns
  squad vs boolq: tau=+0.222 (p=0.477) ns
  squad vs triviaqa: tau=+0.333 (p=0.260) ns
  squad vs hotpotqa: tau=+0.500 (p=0.075) ns
  neuralbridge vs boolq: tau=-0.111 (p=0.761) ns
  neuralbridge vs triviaqa: tau=-0.333 (p=0.260) ns
  neuralbridge vs hotpotqa: tau=+0.056 (p=0.919) ns
  boolq vs triviaqa: tau=+0.333 (p=0.260) ns
  boolq vs hotpotqa: tau=+0.167 (p=0.612) ns
  triviaqa vs hotpotqa: tau=+0.500 (p=0.075) ns

--- I4. Category Means ---

  Category       Conditions                             msmarco      squad neuralbridge      boolq   triviaqa   hotpotqa     Mean
  ---------------------------------------------------------------------------------------------------------------------
  control       

In [12]:
# Cell 22: Summary -- final tables, hypothesis tests, save results
print("=" * 70)
print("SUMMARY -- Prefix LM Exp 05: Hero Run")
print("=" * 70)

class NumpySafeEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.bool_, np.integer)):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)

# --- Replication checks ---
print(f"\n--- Replication Checks ---")

# MS MARCO: d_oracle ~ +0.452, d_random ~ +0.475, structural ~ 105%
cs_mm = dataset_summaries['msmarco']['cond_stats']
d_orc_mm = cs_mm['oracle']['d_correct_vs_bare']
d_rnd_mm = cs_mm['random']['d_correct_vs_bare']
sf_mm = d_rnd_mm / d_orc_mm if d_orc_mm != 0 else float('nan')
print(f"  MS MARCO: d_oracle={d_orc_mm:+.3f} (expected ~+0.452),"
      f" d_random={d_rnd_mm:+.3f} (expected ~+0.475),"
      f" structural={sf_mm:.0%} (expected ~105%)")

# Contrastive sanity: AUC(bare) > 50% on all datasets
print(f"\n  Contrastive sanity (AUC_bare > 50%, random negatives):")
for ds in DATASET_NAMES:
    auc_bare = dataset_summaries[ds]['cond_stats']['bare'].get('auc_rand', 0)
    ok = "PASS" if auc_bare > 0.50 else "FAIL"
    print(f"    {ds}: AUC(bare) = {auc_bare:.1%} [{ok}]")

# --- Final ranked comparison ---
common_conds = BOOLQ_CONDITIONS
surr_conds_ranked = [cn for cn in common_conds if cn not in ('no_doc', 'bare')]

print(f"\n--- Final Surrogate Ranking (d_correct vs bare, mean across 6 datasets) ---\n")
print(f"  {'Rank':>4} {'Condition':<25} {'Category':<14} {'Mean_d':>8}", end="")
for ds in DATASET_NAMES:
    print(f" {ds[:6]:>8}", end="")
print()
print(f"  {'-'*(4+25+14+8+8*len(DATASET_NAMES))}")

# Compute mean d across ALL datasets
mean_d = {}
for cn in surr_conds_ranked:
    mean_d[cn] = np.mean([
        dataset_summaries[ds]['cond_stats'][cn]['d_correct_vs_bare']
        for ds in DATASET_NAMES
    ])

ranked = sorted(surr_conds_ranked, key=lambda c: -mean_d[c])
for rank, cn in enumerate(ranked, 1):
    cat = CONDITION_CATEGORIES[cn]
    print(f"  {rank:>4} {cn:<25} {cat:<14} {mean_d[cn]:>+8.3f}", end="")
    for ds in DATASET_NAMES:
        d = dataset_summaries[ds]['cond_stats'][cn]['d_correct_vs_bare']
        print(f" {d:>+8.3f}", end="")
    print()

# --- Hypothesis Tests ---
print(f"\n--- Hypothesis Tests ---")

# H1: LLM surrogates beat static instructions
d_lq = np.mean([dataset_summaries[ds]['cond_stats']['llm_query']['d_correct_vs_bare']
                 for ds in DATASET_NAMES])
d_inst = np.mean([dataset_summaries[ds]['cond_stats']['instruct']['d_correct_vs_bare']
                   for ds in DATASET_NAMES])
h1 = d_lq > d_inst + 0.02
print(f"  H1 (LLM > static): {'SUPPORTED' if h1 else 'NOT SUPPORTED'}"
      f" (llm_query={d_lq:+.3f} vs instruct={d_inst:+.3f})")

# H2: Token stratification generalizes across all 6 datasets
gradients = [dataset_summaries[ds]['d_gradient_q4_q1'] for ds in DATASET_NAMES]
h2 = sum(1 for g in gradients if g > 0.03) >= 4  # majority
print(f"  H2 (stratification generalizes): {'SUPPORTED' if h2 else 'NOT SUPPORTED'}"
      f" (positive in {sum(1 for g in gradients if g > 0.03)}/6 datasets)")
print(f"      gradients: {dict(zip(DATASET_NAMES, [f'{g:+.3f}' for g in gradients]))}")

# H3: Hard negatives reveal more signal than random negatives
d_discrim_rand = [dataset_summaries[ds].get('d_discrim_rand', 0) for ds in DATASET_NAMES]
d_discrim_hard = [dataset_summaries[ds].get('d_discrim_hard', 0) for ds in DATASET_NAMES]
h3 = np.mean(d_discrim_hard) > np.mean(d_discrim_rand) + 0.02
print(f"  H3 (hard neg > random neg discrimination): {'SUPPORTED' if h3 else 'NOT SUPPORTED'}"
      f" (mean_hard={np.mean(d_discrim_hard):+.3f} vs mean_rand={np.mean(d_discrim_rand):+.3f})")

# H4: Structural fraction varies by dataset characteristics
sfs = [dataset_summaries[ds]['structural_fraction'] for ds in DATASET_NAMES]
sf_range = max(sfs) - min(sfs)
h4 = sf_range > 0.3
print(f"  H4 (structural fraction varies): {'SUPPORTED' if h4 else 'NOT SUPPORTED'}"
      f" (range={sf_range:.0%}, min={min(sfs):.0%}, max={max(sfs):.0%})")

# H5: oracle_plus_vocab consistently beats oracle
h5_wins = 0
for ds in DATASET_NAMES:
    d_opv = dataset_summaries[ds]['cond_stats']['oracle_plus_vocab']['d_correct_vs_bare']
    d_orc = dataset_summaries[ds]['cond_stats']['oracle']['d_correct_vs_bare']
    if d_opv > d_orc + 0.01:
        h5_wins += 1
h5 = h5_wins >= 4
print(f"  H5 (opv > oracle consistently): {'SUPPORTED' if h5 else 'NOT SUPPORTED'}"
      f" (wins {h5_wins}/6 datasets)")

# H6: Pointer instruction competitive with oracle_plus_vocab
d_ptr = np.mean([dataset_summaries[ds]['cond_stats']['pointer']['d_correct_vs_bare']
                  for ds in DATASET_NAMES])
d_opv = np.mean([dataset_summaries[ds]['cond_stats']['oracle_plus_vocab']['d_correct_vs_bare']
                  for ds in DATASET_NAMES])
h6 = abs(d_ptr - d_opv) < 0.05
print(f"  H6 (pointer ~ opv): {'SUPPORTED' if h6 else 'NOT SUPPORTED'}"
      f" (pointer={d_ptr:+.3f} vs opv={d_opv:+.3f})")

# --- Save all results ---
print(f"\n--- Saving Results ---")

# Per-dataset results
for ds in DATASET_NAMES:
    ds_results = {
        'experiment': 'prefix_lm_exp05_hero',
        'dataset': ds,
        'model': MODEL_NAME,
        'n_samples': dataset_summaries[ds]['n_samples'],
        'seed': SEED,
        'conditions': dataset_summaries[ds]['conditions'],
        'condition_categories': CONDITION_CATEGORIES,
        'cond_stats': dataset_summaries[ds]['cond_stats'],
        'structural_fraction': dataset_summaries[ds]['structural_fraction'],
        'd_gradient_q4_q1': dataset_summaries[ds]['d_gradient_q4_q1'],
        'dose_response': dataset_summaries[ds].get('dose_response'),
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    }
    out_path = RESULTS_DIR / ds / 'results.json'
    with open(out_path, 'w') as f:
        json.dump(ds_results, f, indent=2, cls=NumpySafeEncoder)
    print(f"  Saved {out_path}")

# Cross-dataset results
cross_results = {
    'experiment': 'prefix_lm_exp05_hero',
    'model': MODEL_NAME,
    'seed': SEED,
    'datasets': DATASET_NAMES,
    'conditions': ALL_CONDITIONS,
    'condition_categories': CONDITION_CATEGORIES,
    'mean_d_correct': {cn: float(mean_d.get(cn, 0)) for cn in surr_conds_ranked},
    'ranking': ranked,
    'structural_fraction': {
        ds: dataset_summaries[ds]['structural_fraction'] for ds in DATASET_NAMES
    },
    'hypotheses': {
        'H1_llm_beats_static': h1,
        'H2_stratification_generalizes': h2,
        'H3_hard_neg_better': h3,
        'H4_structural_varies': h4,
        'H5_opv_beats_oracle': h5,
        'H6_pointer_matches_opv': h6,
    },
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
}

with open(RESULTS_DIR / 'cross_dataset_results.json', 'w') as f:
    json.dump(cross_results, f, indent=2, cls=NumpySafeEncoder)
print(f"  Saved {RESULTS_DIR / 'cross_dataset_results.json'}")

print(f"\n{'='*70}")
print(f"EXPERIMENT COMPLETE")
print(f"{'='*70}")


SUMMARY -- Prefix LM Exp 05: Hero Run

--- Replication Checks ---
  MS MARCO: d_oracle=+0.452 (expected ~+0.452), d_random=+0.475 (expected ~+0.475), structural=105% (expected ~105%)

  Contrastive sanity (AUC_bare > 50%, random negatives):
    msmarco: AUC(bare) = 83.0% [PASS]
    squad: AUC(bare) = 95.0% [PASS]
    neuralbridge: AUC(bare) = 99.8% [PASS]
    boolq: AUC(bare) = 25.4% [FAIL]
    triviaqa: AUC(bare) = 84.0% [PASS]
    hotpotqa: AUC(bare) = 79.0% [PASS]

--- Final Surrogate Ranking (d_correct vs bare, mean across 6 datasets) ---

  Rank Condition                 Category         Mean_d   msmarc    squad   neural    boolq   trivia   hotpot
  ---------------------------------------------------------------------------------------------------
     1 oracle                    semantic         +0.682   +0.452   +0.823   -0.514   +1.868   +0.861   +0.598
     2 doc_kw10                  extraction       +0.679   +0.461   +0.878   -0.497   +2.267   +0.506   +0.460
     3 llm_keyw

  Saved ../../../results/prefix_lm_exp05/msmarco/results.json
  Saved ../../../results/prefix_lm_exp05/squad/results.json
  Saved ../../../results/prefix_lm_exp05/neuralbridge/results.json
  Saved ../../../results/prefix_lm_exp05/boolq/results.json
  Saved ../../../results/prefix_lm_exp05/triviaqa/results.json
  Saved ../../../results/prefix_lm_exp05/hotpotqa/results.json
  Saved ../../../results/prefix_lm_exp05/cross_dataset_results.json

EXPERIMENT COMPLETE
